Summary: on benchmark le dataloader

In [1]:
from retinotopy import *
welcome()

-----------------------------------------------------------------------------------------
On date 2025-01-05, Running learning on host obiwan.local with device mps, pytorch==2.6.0
-----------------------------------------------------------------------------------------
Welcome on macOS-15.3.1-arm64-arm-64bit


# Loading legacy images

In [2]:
args = Params()
data_set_type = 'full'
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
args.folders = ['train', 'val'] # type of images to use
args

Params(datetag='2025-01-05', loader='data/Imagenet_urls_ILSVRC_2016.json', annotations_animal='data/Animal10k_annotations.json', annotations_train='data/LOC_train_solution.csv', annotations_val='data/LOC_val_solution.csv', folders=['train', 'val'], tasks=['animal', 'dog', 'cat', 'bird'], image_size=224, num_epochs=2, n_train_stop=0, seed=1998, batch_size=75, batch_size_val=75, lr=0.00015, momentum=0.06, beta2=0, rs_min=0.0, rs_max=-5.0, do_polar=True, do_raw=False, do_translate=False, do_resize=True, do_mask=True, do_scratch=False, do_rotation=False, do_rot_train=False, resolution=(11, 11), size_ratio=0.1, do_saccade=False, do_zoom=False, method='valid', saccade_type='multi', normalize=True, verbose=False)

In [3]:
%%timeit -n1
args.folders = ['train', 'val'] # type of images to use
dataloaders = datasets_transforms(args)
len(dataloaders['train']), len(dataloaders['train'].dataset)

Loaded 1281100 images under train
Loaded 50000 images under val
Loaded 1281100 images under train
Loaded 50000 images under val
Loaded 1281100 images under train
Loaded 50000 images under val
Loaded 1281100 images under train
Loaded 50000 images under val
Loaded 1281100 images under train
Loaded 50000 images under val
Loaded 1281100 images under train
Loaded 50000 images under val
Loaded 1281100 images under train
Loaded 50000 images under val
14.5 s ± 2.49 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
%%timeit -n1
args.folders = ['val'] # type of images to use
dataloaders = datasets_transforms(args)
len(dataloaders['val']), len(dataloaders['val'].dataset)

Loaded 50000 images under val
Loaded 50000 images under val
Loaded 50000 images under val
Loaded 50000 images under val
Loaded 50000 images under val
Loaded 50000 images under val
Loaded 50000 images under val
212 ms ± 13.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Benchmarking different methods for the dataloader:

In [5]:
for num_workers_ in [0, 1, 2, 5, 8]: # , 16 , 32
    for batch_size_ in [1, 4, 16, 32, 128, 256, 512]:
        for pin_memory_ in [True, False]:
            args = Params()
            args.batch_size = batch_size_
            args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
            args.folders = ['val'] # type of images to use            
        
            dataloaders = datasets_transforms(args, pin_memory=pin_memory_, num_workers=num_workers_, verbose=False)
            tic = time.time()
            i_image, i_image_max = 0, 4096
            for i_step, (images, labels) in enumerate(dataloaders['val']):
                    images, labels = images.to(device), labels.to(device)
                    i_image += len(images)
                    if i_image > i_image_max:
                        break

            toc = time.time()
            print(f'{pin_memory_=} \t {num_workers_=} \t {batch_size_=:04d} \t Loading time for {i_image_max} images \t {toc-tic:.1f} s')  

pin_memory_=True 	 num_workers_=0 	 batch_size_=0001 	 Loading time for 4096 images 	 26.3 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0001 	 Loading time for 4096 images 	 28.2 s
pin_memory_=True 	 num_workers_=0 	 batch_size_=0004 	 Loading time for 4096 images 	 25.0 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0004 	 Loading time for 4096 images 	 24.7 s
pin_memory_=True 	 num_workers_=0 	 batch_size_=0016 	 Loading time for 4096 images 	 23.1 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0016 	 Loading time for 4096 images 	 22.8 s
pin_memory_=True 	 num_workers_=0 	 batch_size_=0032 	 Loading time for 4096 images 	 21.8 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0032 	 Loading time for 4096 images 	 23.4 s
pin_memory_=True 	 num_workers_=0 	 batch_size_=0128 	 Loading time for 4096 images 	 21.9 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0128 	 Loading time for 4096 images 	 22.4 s
pin_memory_=True 	 num_workers_=0 	 batch_size_=0256 	 Loading ti

In [10]:
model_filename = f'cached_data/{datetag}_full_resnet101_retino.pt'
model = load_model(model_name='resnet101', model_path=model_filename, do_scratch=False, do_circular=False, verbose=True).to(device)

N_test = 2**8
for num_workers_ in [0, 1, 2, 5, 8]: # , 16 , 32
    for batch_size_ in [1, 4, 16, 32, 64, 128, 256, 512]:
        for pin_memory_ in [True, False]: # [False]: #
            args = Params()
            args.batch_size_val = batch_size_
            args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
            args.folders = ['val'] # type of images to use            
        
            dataloaders = datasets_transforms(args, pin_memory=pin_memory_, num_workers=num_workers_, verbose=False)
            tic = time.time()
            for i_step, (images, labels) in enumerate(dataloaders['val']):
                    images, labels = images.to(device), labels.to(device)
                    with torch.no_grad():
                        outputs = model(images)
                    if i_step > N_test/batch_size_: break
            toc = time.time()
            print(f'{pin_memory_=} \t\t {num_workers_=} \t\t {batch_size_=:03d} \t\t Elapsed time per image: {1000*(toc-tic)/N_test:.1f} ms')  

loading .... cached_data/2025-01-05_full_resnet101_retino.pt
pin_memory_=True 		 num_workers_=0 		 batch_size_=001 		 Elapsed time per image: 37.6 ms
pin_memory_=False 		 num_workers_=0 		 batch_size_=001 		 Elapsed time per image: 38.7 ms
pin_memory_=True 		 num_workers_=0 		 batch_size_=004 		 Elapsed time per image: 26.1 ms
pin_memory_=False 		 num_workers_=0 		 batch_size_=004 		 Elapsed time per image: 15.4 ms
pin_memory_=True 		 num_workers_=0 		 batch_size_=016 		 Elapsed time per image: 13.5 ms
pin_memory_=False 		 num_workers_=0 		 batch_size_=016 		 Elapsed time per image: 11.4 ms
pin_memory_=True 		 num_workers_=0 		 batch_size_=032 		 Elapsed time per image: 16.2 ms
pin_memory_=False 		 num_workers_=0 		 batch_size_=032 		 Elapsed time per image: 12.7 ms
pin_memory_=True 		 num_workers_=0 		 batch_size_=064 		 Elapsed time per image: 15.7 ms
pin_memory_=False 		 num_workers_=0 		 batch_size_=064 		 Elapsed time per image: 13.1 ms
pin_memory_=True 		 num_workers_=0 		 batch_

RuntimeError: Broken pipe